> **Research-source reading copy.** Saved outputs and machine-specific metadata were removed for sharing. These original workflow cells require the private raw/source-evidence workspace and database, which are intentionally not bundled. For an executable portable result replay, use `python scripts/replay_saved_study.py` from the repository root. See README and docs/REPRODUCIBILITY.md.

# 01 — BSP USD/PHP exploratory analysis

**Purpose:** understand the data, translate exchange-rate movements into hypothetical financial impact, and establish a defensible foundation for risk modelling.

**Study:** frozen BSP observations from January 2020 through September 2026; fixed, hypothetical, unhedged long USD1 million; reporting currency PHP. The USD quantity stays constant. Gains and losses below are calculated valuation changes, not executed trades or actual bank results.

**Reading guide:** each section states a question, shows the calculation or chart, then records the finding and its limits. Rates are PHP per USD; simple percentage changes are displayed in percent, not decimals. Standard deviations of those percentage changes are in percentage points and are not annualized.

Original source dates and missing rates are preserved. Selected values match the USD-row **PHIL. PESO EQUIVALENT** field of BSP bulletins; this is different from the separately named BSP Reference Rate and PDS Closing Rate. Source dates are not certified market dates, and historical availability timestamps are unverified. Use **observation-to-observation**, not verified market-close daily, terminology.

**Status:** agreed EDA and closeout diagnostics completed. VaR implementation and backtesting have not started in this notebook. See [data pipeline](../DATA_PIPELINE.md) and [modelling contract v1.0](../MODELLING_CONTRACT.md).

**Research context:** this guided study examines the 2022 case, computed outputs and reviewed evidence. AI assistance supported code, source checks and editing. The contribution record documents that assistance.

**Contents:** 1 Setup · 2 Coverage and levels · 3 2022 case study · 4 Interval P&L · 5 Drawdown · 6 Rolling volatility · 7 Distribution · 8 Calendar eligibility · 9 Yearly comparison · 10 Warm-up coverage · 11 Repeated values · 12 Dependence · 13 Training-window composition · 14 Closeout.


## 1. Confirm read-only database access

**Question:** are we using the project environment and intended database?

Run from the project root or its `notebooks` directory using the project `.venv` kernel. This cell imports the tools and lists raw/staging tables without changing the database.

In [ ]:
from pathlib import Path
import sys

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

database = project_root / "traded_risk.duckdb"
if not database.is_file():
    raise FileNotFoundError(f"Open the project root or notebooks folder. Database not found: {database}")

print("Python:", sys.executable)
print("Database:", database)

with duckdb.connect(str(database), read_only=True) as con:
    tables = con.execute("""
        SELECT table_schema, table_name
        FROM information_schema.tables
        WHERE table_schema IN ('raw', 'staging')
        ORDER BY table_schema, table_name;
    """).df()

tables


**Access check:** both `raw.bsp_usd_php` and `staging.bsp_usd_php` are available. This establishes access, not source accuracy or completed analysis.

## 2. Establish coverage and inspect rate levels

**Question:** what period and how many usable rates does this frozen snapshot contain?

Keep calendar rows, including NULL rates, in `fx`. Filtering is scoped to the frozen workbook hash.

In [ ]:
query = """
    SELECT
        source_date,
        usd_php_rate,
        rate_status
    FROM staging.bsp_usd_php
    WHERE snapshot_sha256 = ?
    ORDER BY source_date;
"""

snapshot_sha256 = (
    "3cf2fd962d1fbd9c4ecc59f56bb71204aa5a29d5f0ee666faa956835532c6848"
)

with duckdb.connect(str(database), read_only=True) as con:
    fx = con.execute(query, [snapshot_sha256]).df()

fx["source_date"] = pd.to_datetime(fx["source_date"])

pd.DataFrame({
    "metric": [
        "First source date",
        "Last source date",
        "Total observations",
        "Available numeric rates",
        "Missing rates",
    ],
    "value": [
        fx["source_date"].min().date(),
        fx["source_date"].max().date(),
        len(fx),
        fx["usd_php_rate"].notna().sum(),
        fx["usd_php_rate"].isna().sum(),
    ],
})


**Finding:** 2,465 calendar-date rows cover 2020-01-01 to 2026-09-30; 1,648 rates are numeric and 817 are missing source markers. Expected publication gaps are included among the missing values, so missing does not automatically mean erroneous.

### 2.1 Full-period rate levels

**Question:** which broad periods merit investigation? A rising PHP-per-USD quote increases the PHP value of the fixed long-USD position. Missing observations remain breaks in the line.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))

ax.plot(
    fx["source_date"],
    fx["usd_php_rate"],
    color="steelblue",
    linewidth=1,
)

ax.set_title("BSP USD/PHP Published Rates | 2020–September 2026")
ax.set_xlabel("BSP source date")
ax.set_ylabel("PHP per US$1")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


**Observation:** the series shows a decline into 2021, a pronounced 2022 rise/reversal, and higher levels near the end of the study. These are visual descriptions only. High levels do not imply high volatility, establish causes, or predict the next direction.

## 3. Investigate the rise and reversal

**Question:** what happened within a year that appears positive from start to finish?

The 2022 case illustrates the rise and reversal visible in the full-period series. It was examined retrospectively, so it provides an explanatory example with hindsight.

In [ ]:
fx_2022 = fx.loc[
    fx["source_date"].between("2022-01-01", "2022-12-31")
].copy()

fig, ax = plt.subplots(figsize=(12, 4.5))

ax.plot(
    fx_2022["source_date"],
    fx_2022["usd_php_rate"],
    color="steelblue",
    linewidth=1.3,
)

ax.set_title("Investigating the 2022 USD/PHP Rise and Reversal")
ax.set_xlabel("BSP source date")
ax.set_ylabel("PHP per US$1")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


### 3.1 Identify exact observed endpoints

Use available rates for endpoint selection while retaining the original calendar-date data separately. List every date tied at the highest observed rate.

In [ ]:
valid_2022 = (
    fx_2022.dropna(subset=["usd_php_rate"])
    .sort_values("source_date")
)

first = valid_2022.iloc[0]
last = valid_2022.iloc[-1]

peak_rate = valid_2022["usd_php_rate"].max()
peak_dates = valid_2022.loc[
    valid_2022["usd_php_rate"].eq(peak_rate),
    "source_date"
]

pd.DataFrame({
    "point": [
        "First available observation",
        "Highest observed rate",
        "Last available observation",
    ],
    "source_date": [
        first["source_date"].date(),
        ", ".join(peak_dates.dt.strftime("%Y-%m-%d")),
        last["source_date"].date(),
    ],
    "php_per_usd": [
        first["usd_php_rate"],
        peak_rate,
        last["usd_php_rate"],
    ],
})


### 3.2 Compare endpoint changes

These are multi-month quote changes. They are not daily returns, volatility, or the reciprocal percentage change in PHP's value.

In [ ]:
start_rate = first["usd_php_rate"]
end_rate = last["usd_php_rate"]

pd.DataFrame({
    "interval": ["Start → peak", "Peak → year-end", "Start → year-end"],
    "change_php_per_usd": [
        peak_rate - start_rate,
        end_rate - peak_rate,
        end_rate - start_rate,
    ],
    "change_percent": [
        (peak_rate / start_rate - 1) * 100,
        (end_rate / peak_rate - 1) * 100,
        (end_rate / start_rate - 1) * 100,
    ],
}).round(2)


**Finding:** 50.974 on January 3 rose to an observed peak of 58.994 on October 11 and ended at 56.120 on December 29. Start-to-peak was +15.73%, peak-to-year-end -4.87%, and first-to-last +10.10%. At fixed USD1m, the respective PHP changes are +8.020m, -2.874m and +5.146m. The peak was identified after observing the year.

## 4. Calculate original interval changes and financial impact

**Question:** how much did the rate change between consecutive available observations?

Calculate changes across the full series before selecting 2022. Preserve each interval's actual endpoints and calendar gap. Dropping missing rates here does not fill them or establish a one-trading-day horizon.

In [ ]:
# Work on a copy; preserve the original calendar-date data and database.
fx_changes = (
    fx.loc[fx["usd_php_rate"].notna(), ["source_date", "usd_php_rate"]]
    .sort_values("source_date")
    .copy()
)

# Pair each available rate with the preceding available observation.
fx_changes["previous_source_date"] = fx_changes["source_date"].shift(1)
fx_changes["previous_rate"] = fx_changes["usd_php_rate"].shift(1)

# Measure calendar spacing; this is not a verified trading-day horizon.
fx_changes["gap_calendar_days"] = (
    fx_changes["source_date"] - fx_changes["previous_source_date"]
).dt.days.astype("Int64")

# Keep the rate difference and simple percentage change unrounded.
fx_changes["change_php_per_usd"] = (
    fx_changes["usd_php_rate"] - fx_changes["previous_rate"]
)

fx_changes["change_percent"] = (
    fx_changes["usd_php_rate"] / fx_changes["previous_rate"] - 1
) * 100

# Display the first ten changes ending in the selected 2022 case study.
fx_changes.loc[
    fx_changes["source_date"].between("2022-01-01", "2022-12-31"),
    [
        "previous_source_date",
        "source_date",
        "gap_calendar_days",
        "previous_rate",
        "usd_php_rate",
        "change_php_per_usd",
        "change_percent",
    ],
].head(10).round({"previous_rate": 4, "usd_php_rate": 4, "change_php_per_usd": 4, "change_percent": 4})


**Reading:** a three-calendar-day gap can be Friday to Monday; do not divide its change by three to invent daily returns. The first interval ending in 2022 starts on December 31, 2021. These original intervals are retained throughout later eligibility filtering.

### 4.1 Rank the largest observed 2022 interval losses

For a fixed quantity, PHP P&L = USD quantity × change in PHP-per-USD rate. Negative P&L means a valuation loss for this long-USD position. Rank by peso impact and display interval gaps.

In [ ]:
# Select changes ending in 2022
changes_2022 = fx_changes.loc[
    fx_changes["source_date"].between("2022-01-01", "2022-12-31")
].copy()

# PHP valuation change for our fixed hypothetical USD 1 million
usd_position = 1_000_000

changes_2022["position_pnl_php"] = (
    changes_2022["change_php_per_usd"] * usd_position
)

# Five largest observation-to-observation losses
changes_2022.nsmallest(5, "position_pnl_php")[
    [
        "previous_source_date",
        "source_date",
        "gap_calendar_days",
        "previous_rate",
        "usd_php_rate",
        "change_percent",
        "position_pnl_php",
    ]
].round({"previous_rate": 4, "usd_php_rate": 4, "change_percent": 4, "position_pnl_php": 4})


### 4.2 View interval P&L through 2022

Each bar is one original observation interval, not cumulative P&L. Green bars are gains and red bars losses. The vertical axis is PHP thousands; -500 means a PHP500,000 loss.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Express peso amounts in thousands for easier reading
pnl_thousands = changes_2022["position_pnl_php"] / 1_000

# Green = gain; red = loss
colors = [
    "seagreen" if pnl >= 0 else "indianred"
    for pnl in pnl_thousands
]

ax.bar(
    changes_2022["source_date"],
    pnl_thousands,
    color=colors,
    width=0.8,
)

# Zero separates gains from losses
ax.axhline(0, color="black", linewidth=0.8)

ax.set_title(
    "2022 Observation-to-Observation FX P&L | Fixed USD 1 Million"
)
ax.set_xlabel("BSP source date")
ax.set_ylabel("PHP valuation change (₱ thousands)")
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()


**Observation:** deeper losses are visually prominent late in 2022, but gains also occur. Include both when assessing net P&L. Nearby red bars suggest accumulated declines for this position; the chart alone does not prove volatility clustering or a news cause.

### 4.3 Identify both extremes

Select the biggest loss and gain, retaining ties. A large gain is not a loss-side VaR exception for this long position. No VaR has yet been calculated.

In [ ]:
# Find the largest loss and largest gain
# Keep all matching rows if there are ties
pnl = changes_2022["position_pnl_php"]

biggest_loss = changes_2022.loc[pnl.eq(pnl.min())].copy()
biggest_loss["movement"] = "Biggest loss"

biggest_gain = changes_2022.loc[pnl.eq(pnl.max())].copy()
biggest_gain["movement"] = "Biggest gain"

extreme_movements = pd.concat(
    [biggest_loss, biggest_gain],
    ignore_index=True,
)

extreme_movements[
    [
        "movement",
        "previous_source_date",
        "source_date",
        "gap_calendar_days",
        "previous_rate",
        "usd_php_rate",
        "change_percent",
        "position_pnl_php",
    ]
].round({"previous_rate": 4, "usd_php_rate": 4, "change_percent": 4, "position_pnl_php": 4})


**Finding:** the largest 2022 loss was PHP774,000 over November 11–14 (three calendar days); the largest gain was PHP727,000 over December 28–29 (one day). Both end in 2022. They are historical valuation extremes, not maximum possible future losses or established VaR exceptions.

## 5. Measure drawdown from the running peak

**Question:** how far did position value fall below its highest value reached so far?

The running peak uses observations only through each point and starts at the first available 2022 observation. Drawdown is current value minus the running peak; zero means at a running high. Missing dates remain gaps in the chart.

In [ ]:
# Available observations within 2022
drawdown_2022 = (
    fx_2022.loc[
        fx_2022["usd_php_rate"].notna(),
        ["source_date", "usd_php_rate"],
    ]
    .sort_values("source_date")
    .copy()
)

# PHP value of our fixed USD 1 million position
drawdown_2022["position_value_php"] = (
    drawdown_2022["usd_php_rate"] * usd_position
)

# Track the within-2022 peak using no later observations.
drawdown_2022["running_peak_php"] = (
    drawdown_2022["position_value_php"].cummax()
)

# Decline from that running peak
drawdown_2022["drawdown_php"] = (
    drawdown_2022["position_value_php"]
    - drawdown_2022["running_peak_php"]
)

# Return to the calendar-date grid so missing rates remain gaps
drawdown_plot = fx_2022[["source_date"]].merge(
    drawdown_2022[["source_date", "drawdown_php"]],
    on="source_date",
    how="left",
)

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    drawdown_plot["source_date"],
    drawdown_plot["drawdown_php"] / 1_000_000,
    color="indianred",
    linewidth=1.4,
)

ax.axhline(0, color="black", linewidth=0.8)

ax.set_title(
    "2022 Drawdown from Running Peak | Fixed USD 1 Million"
)
ax.set_xlabel("BSP source date")
ax.set_ylabel("Drawdown (₱ millions)")
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()


### 5.1 Identify peak, trough and observed recovery

Find the first deepest trough; if its preceding peak is repeated, select the latest occurrence before the trough. Recovery means reaching that peak again within the remaining 2022 observations.

In [ ]:
# Use chronological observations with a simple row index
dd = drawdown_2022.sort_values("source_date").reset_index(drop=True)

# Deepest drawdown; use the first occurrence if tied
trough_index = dd["drawdown_php"].idxmin()
trough = dd.loc[trough_index]

# Find the peak reached before that trough
history_to_trough = dd.loc[:trough_index]
peak_value = trough["running_peak_php"]

# If the same peak occurred several times, use the latest before the trough
peak = history_to_trough.loc[
    history_to_trough["position_value_php"].eq(peak_value)
].iloc[-1]

# Check whether the position recovered to that peak later in 2022
recovery_rows = dd.loc[
    (dd["source_date"] > trough["source_date"])
    & (dd["position_value_php"] >= peak_value)
]

recovery_date = (
    recovery_rows.iloc[0]["source_date"].date()
    if not recovery_rows.empty
    else "Not observed within 2022"
)

maximum_drawdown = pd.DataFrame({
    "peak_date": [peak["source_date"].date()],
    "trough_date": [trough["source_date"].date()],
    "peak_value_php": [peak_value],
    "trough_value_php": [trough["position_value_php"]],
    "maximum_drawdown_php": [trough["drawdown_php"]],
    "maximum_drawdown_percent": [
        (trough["position_value_php"] / peak_value - 1) * 100
    ],
    "peak_to_trough_calendar_days": [
        (trough["source_date"] - peak["source_date"]).days
    ],
    "recovery_date": [recovery_date],
})

maximum_drawdown.round(4)


### 5.2 Apply the illustrative 3% review threshold

The retrospective exploration uses an illustrative 3% running-peak drawdown flag. A breach means drawdown at or below -3%. This is a review illustration, not a calibrated bank limit or automatic hedge/sell instruction.

In [ ]:
# Analysis copy; original data stays unchanged
dd_flags = drawdown_2022.sort_values("source_date").copy()

# Percentage decline from the highest value reached so far
dd_flags["drawdown_percent"] = (
    dd_flags["drawdown_php"]
    / dd_flags["running_peak_php"]
) * 100

# Illustrative, hindsight-selected review trigger: drawdown at or below -3%.
threshold_percent = 3.0

dd_flags["drawdown_review_flag"] = (
    dd_flags["drawdown_percent"] <= -threshold_percent
)

flagged_rows = dd_flags.loc[dd_flags["drawdown_review_flag"]]

if flagged_rows.empty:
    print("No 3% drawdown breach observed within 2022.")
else:
    first_breach = flagged_rows.iloc[0]

    first_breach_summary = pd.DataFrame({
        "first_breach_source_date": [
            first_breach["source_date"].date()
        ],
        "running_peak_php": [
            first_breach["running_peak_php"]
        ],
        "review_trigger_value_php": [
            first_breach["running_peak_php"]
            * (1 - threshold_percent / 100)
        ],
        "observed_position_value_php": [
            first_breach["position_value_php"]
        ],
        "drawdown_php": [
            first_breach["drawdown_php"]
        ],
        "drawdown_percent": [
            first_breach["drawdown_percent"]
        ],
    })

    display(first_breach_summary.round(4))


### 5.3 Locate the first observed breach visually

Plot percentage drawdown so the -3% threshold is directly comparable. The marked source date is the first observed crossing in this series, not an intraday crossing or verified release timestamp.

In [ ]:
# Preserve missing dates as gaps in the chart
flag_plot = fx_2022[["source_date"]].merge(
    dd_flags[["source_date", "drawdown_percent"]],
    on="source_date",
    how="left",
)

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    flag_plot["source_date"],
    flag_plot["drawdown_percent"],
    color="steelblue",
    linewidth=1.4,
    label="Observed drawdown",
)

ax.axhline(0, color="black", linewidth=0.8)

ax.axhline(
    -threshold_percent,
    color="darkorange",
    linestyle="--",
    label="Illustrative −3% review threshold",
)

# Mark the first observed breach
breaches = dd_flags.loc[dd_flags["drawdown_review_flag"]]

if not breaches.empty:
    breach = breaches.iloc[0]
    breach_date = breach["source_date"]
    breach_percent = breach["drawdown_percent"]

    ax.scatter(
        breach_date,
        breach_percent,
        color="crimson",
        s=70,
        zorder=5,
        label="First observed breach",
    )

    ax.axvline(
        breach_date,
        color="crimson",
        linestyle=":",
        alpha=0.5,
    )

    ax.annotate(
        f"{breach_date:%d %b %Y}\n"
        f"{breach_percent:.2f}% | "
        f"₱{breach['drawdown_php'] / 1_000_000:.3f}m",
        xy=(breach_date, breach_percent),
        xytext=(-150, -65),
        textcoords="offset points",
        arrowprops=dict(arrowstyle="->", color="crimson"),
        fontsize=10,
    )

ax.set_title("2022 Drawdown and First Observed 3% Breach | USD 1 Million")
ax.set_xlabel("BSP source date")
ax.set_ylabel("Drawdown from running peak (%)")
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="lower left")

plt.tight_layout()
plt.show()


### 5.4 Interpretation: a positive year can contain a substantial decline

Maximum observed within-2022 drawdown was **PHP3.864m (6.5498%)**, from the October 11 peak (PHP58.994m) to the December 23 trough (PHP55.130m), over 73 calendar days. This does not mean every intervening observation was a loss. Recovery to the peak was not observed within 2022; later recovery is outside this calculation.

The illustrative 3% flag first triggered on **November 24**, at -3.0359% / -PHP1.791m. It identifies an already-observed decline, not knowledge that a larger decline would follow. No economic loss-avoidance or operational alert feasibility is established.

**Key distinction:** first-to-last P&L was +PHP5.146m, while peak-to-trough drawdown was -PHP3.864m. Different reference points answer different questions. The report links each flag to evidence and a review note. The threshold is illustrative and was examined retrospectively. Operational calibration would require a documented loss tolerance and review capacity.


## 6. Describe rolling variability

**Question:** when were recent percentage changes more or less variable?

Compute sample standard deviation over the last 20 original changes, including the current one. Early-2022 windows may use late-2021 history. Each complete window requires 21 rate observations. This is descriptive rolling volatility, not a prior forecast or annualized risk estimate.

In [ ]:
# Use the full history so early 2022 has prior observations available
volatility = fx_changes.sort_values("source_date").copy()

window = 20

volatility["rolling_volatility_pct"] = (
    volatility["change_percent"]
    .rolling(window=window, min_periods=window)
    .std(ddof=1)
)

# Preserve missing dates as gaps when plotting 2022
volatility_plot = fx_2022[["source_date"]].merge(
    volatility[["source_date", "rolling_volatility_pct"]],
    on="source_date",
    how="left",
)

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    volatility_plot["source_date"],
    volatility_plot["rolling_volatility_pct"],
    color="darkviolet",
    linewidth=1.5,
)

ax.set_title(
    "2022 Rolling Volatility | Last 20 Observation-to-Observation Changes"
)
ax.set_xlabel("BSP source date")
ax.set_ylabel("Standard deviation of percentage changes (%)")
ax.set_ylim(bottom=0)
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()


**Interpretation:** higher values mean the last 20 percentage changes were more dispersed. Large gains and losses can both increase volatility. Overlapping windows share 19 changes, so persistence in this line alone is not evidence of volatility clustering. Calendar gaps remain heterogeneous.

### 6.1 Identify complete high- and low-volatility windows

Select windows ending within 2022, but retain their actual starting dates. The chart's end-date value summarizes 20 changes, not just that date's movement.

In [ ]:
# Retain full history so each 20-change window has its true starting observation.
vol_windows = (
    volatility.sort_values("source_date")
    .reset_index(drop=True)
    .copy()
)

# Twenty changes require twenty-one endpoint observations.
vol_windows["window_start_date"] = (
    vol_windows["previous_source_date"].shift(window - 1)
)

vol_windows["window_end_date"] = vol_windows["source_date"]

# Select complete windows ending within 2022
vol_2022 = vol_windows.loc[
    vol_windows["source_date"].between("2022-01-01", "2022-12-31")
    & vol_windows["rolling_volatility_pct"].notna()
].copy()

if vol_2022.empty:
    print("No complete rolling windows ending within 2022.")
else:
    highest = vol_2022.loc[
        vol_2022["rolling_volatility_pct"].eq(
            vol_2022["rolling_volatility_pct"].max()
        )
    ].copy()
    highest["point"] = "Highest rolling volatility"

    lowest = vol_2022.loc[
        vol_2022["rolling_volatility_pct"].eq(
            vol_2022["rolling_volatility_pct"].min()
        )
    ].copy()
    lowest["point"] = "Lowest rolling volatility"

    volatility_extremes = pd.concat(
        [highest, lowest],
        ignore_index=True,
    )

    volatility_extremes["changes_in_window"] = window
    volatility_extremes["window_calendar_days"] = (
        volatility_extremes["window_end_date"]
        - volatility_extremes["window_start_date"]
    ).dt.days

    display(
        volatility_extremes[
            [
                "point",
                "window_start_date",
                "window_end_date",
                "changes_in_window",
                "window_calendar_days",
                "rolling_volatility_pct",
            ]
        ].round({"rolling_volatility_pct": 6})
    )


**Finding:** highest rolling volatility was 0.487180 percentage points over November 28–December 29 (20 changes/31 calendar days); lowest was 0.160868 over May 16–June 13 (20 changes/28 days). These are selected extrema, not independently chosen comparison periods.

### 6.2 Compare variability with net P&L

Net P&L compares window-end value with window-start value; volatility describes the changes along the way. These hindsight-selected windows illustrate the distinction and do not establish a general volatility-to-loss relationship.

In [ ]:
# Lookup available rates by source date
rate_lookup = (
    fx.loc[fx["usd_php_rate"].notna()]
    .set_index("source_date")["usd_php_rate"]
)

window_comparison = volatility_extremes[
    [
        "point",
        "window_start_date",
        "window_end_date",
        "changes_in_window",
        "window_calendar_days",
        "rolling_volatility_pct",
    ]
].copy()

# Rates at the two endpoints
window_comparison["start_rate"] = (
    window_comparison["window_start_date"].map(rate_lookup)
)

window_comparison["end_rate"] = (
    window_comparison["window_end_date"].map(rate_lookup)
)

# Net PHP valuation change for the fixed USD 1 million
window_comparison["net_pnl_php"] = (
    window_comparison["end_rate"]
    - window_comparison["start_rate"]
) * usd_position

window_comparison["net_change_percent"] = (
    window_comparison["end_rate"]
    / window_comparison["start_rate"]
    - 1
) * 100

display(window_comparison.round({"rolling_volatility_pct": 4, "start_rate": 4, "end_rate": 4, "net_pnl_php": 4, "net_change_percent": 4}))


**Finding:** the higher-volatility window had net P&L of -PHP633,000 (-1.1154%); the lower-volatility window had +PHP517,000 (+0.9860%). This does not mean high volatility always causes losses. Net P&L describes endpoints; drawdown describes decline from a prior peak; volatility describes variability.

## 7. Inspect the distribution of 2022 changes

**Question:** which movements were common, and what did the extremes look like?

Histogram bars count intervals in percentage-change ranges, not dates. Left-tail changes are losses for long USD; right-tail changes are gains. Shape depends partly on binning and does not establish normality.

In [ ]:
# Available observation-to-observation percentage changes ending in 2022
returns_2022 = changes_2022["change_percent"].dropna()

fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(
    returns_2022,
    bins=25,
    color="steelblue",
    edgecolor="white",
)

ax.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
    label="Zero change",
)

ax.set_title("2022 Distribution of Observation-to-Observation Changes")
ax.set_xlabel("Change in PHP-per-USD rate (%)")
ax.set_ylabel("Number of observation intervals")
ax.legend()
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()


### 7.1 Summarize skewness and excess kurtosis

Skewness describes asymmetry; excess kurtosis measures the fourth standardized moment relative to the normal benchmark. Pandas `kurt()` already returns excess kurtosis: do not subtract three again. These are finite-sample estimates, not future loss probabilities.

In [ ]:
# Percentage changes for observation intervals ending in 2022
changes = changes_2022["change_percent"].dropna()

distribution_summary = pd.DataFrame({
    "metric": [
        "Number of changes",
        "Mean change (%)",
        "Median change (%)",
        "Skewness",
        "Excess kurtosis",
    ],
    "value": [
        changes.count(),
        changes.mean(),
        changes.median(),
        changes.skew(),
        changes.kurt(),
    ],
})

display(distribution_summary.round(4))


**Finding:** 245 changes ending in 2022 have mean +0.0414%, median +0.0115%, skewness 0.0855 and excess kurtosis 1.6976. The histogram appears roughly symmetric, with meaningful extremes on both sides. Normality has not been established. Positive excess kurtosis raises concern about relying solely on a normal assumption but does not prove a particular 99% left-tail VaR would be underestimated. Historical simulation remains a starting method with sparse tail support, not a proven superior model.

## 8. Audit calendar gaps before modelling

**Question:** which durations occur between available observations?

Calendar days are not verified trading days or equal economic holding periods. Keep the first observation out of the return count because it has no preceding rate.

In [ ]:
# Exclude the first observation, which has no prior rate
interval_audit = fx_changes.loc[
    fx_changes["change_percent"].notna()
].copy()

gap_summary = (
    interval_audit
    .groupby("gap_calendar_days")
    .agg(
        number_of_intervals=("change_percent", "size"),
        largest_decline_pct=("change_percent", "min"),
        largest_increase_pct=("change_percent", "max"),
    )
    .reset_index()
)

gap_summary["share_of_intervals_pct"] = (
    gap_summary["number_of_intervals"]
    / len(interval_audit)
    * 100
)

display(
    gap_summary[
        [
            "gap_calendar_days",
            "number_of_intervals",
            "share_of_intervals_pct",
            "largest_decline_pct",
            "largest_increase_pct",
        ]
    ].round(4)
)


**Finding:** 1,647 original intervals have gaps of one to six calendar days. A long gap is not automatically an error, and a one-day gap is not automatically verified market-close timing. Eligibility depends on intervening-date evidence and the declared modelling interpretation.

### 8.1 Match intervening dates to calendar evidence

Inspect every date strictly between the two endpoints. Adjacent dates or gaps explained entirely by weekends/verified national nonworking days are calendar candidates. An unresolved or absent evidence record triggers review. Candidate status is not final verification of market timing.

In [ ]:
# Read calendar evidence from staging
with duckdb.connect(str(database), read_only=True) as con:
    calendar = con.execute(
        """
        SELECT source_date, rate_status, calendar_status
        FROM staging.bsp_usd_php
        WHERE snapshot_sha256 = ?
        ORDER BY source_date
        """,
        [snapshot_sha256],
    ).df()

calendar["source_date"] = pd.to_datetime(calendar["source_date"])

# One calendar record per date is required
assert not calendar["source_date"].duplicated().any(), (
    "Duplicate calendar dates found; review before proceeding."
)

calendar = calendar.set_index("source_date")

explained_statuses = {
    "weekend",
    "verified_national_nonworking_day",
}

classified_intervals = []

for row in interval_audit.itertuples(index=False):
    start = row.previous_source_date
    end = row.source_date

    # Dates strictly between the two available observations
    between_dates = pd.date_range(
        start=start,
        end=end,
        freq="D",
        inclusive="neither",
    )

    evidence = calendar.reindex(between_dates)

    # Every intervening date must have an explained source-missing record
    explained = (
        evidence["rate_status"].eq("source_missing")
        & evidence["calendar_status"].isin(explained_statuses)
    )

    review_dates = evidence.index[~explained]

    if len(between_dates) == 0:
        status = "candidate"
        reason = "No intervening calendar dates"
    elif explained.all():
        status = "candidate"
        reason = "Only explained nonpublication calendar gaps"
    else:
        status = "review"
        reason = "Unresolved or incomplete intervening-date evidence"

    classified_intervals.append({
        "previous_source_date": start,
        "source_date": end,
        "gap_calendar_days": row.gap_calendar_days,
        "change_percent": row.change_percent,
        "calendar_eligibility": status,
        "reason": reason,
        "review_dates": ", ".join(
            review_dates.strftime("%Y-%m-%d")
        ),
    })

interval_calendar_audit = pd.DataFrame(classified_intervals)

# Summary
display(
    interval_calendar_audit
    .groupby(["calendar_eligibility", "reason"])
    .size()
    .reset_index(name="number_of_intervals")
)

# Show intervals requiring review
display(
    interval_calendar_audit.loc[
        interval_calendar_audit["calendar_eligibility"].eq("review")
    ].round({"change_percent": 4})
)


**Finding:** 1,263 intervals have no intervening dates; 377 cross only explained calendar gaps; seven require review. Thus 1,640 are calendar candidates. Review does not mean incorrect endpoint values. The contract excludes the seven from the main run and preserves a separately labelled inclusion sensitivity; it never bridges excluded returns.

## 9. Compare years rather than generalizing from 2022

**Question:** was 2022 uniquely extreme in the study?

Use all original changes, including review intervals, and assign each interval to its ending year. Yearly volatility is sample standard deviation of changes, not annualized volatility. The first interval may start in the previous year; 2026 is partial through September 30.

In [ ]:
# Use the interval table with calendar classifications
yearly_data = interval_calendar_audit.copy()

# Assign each interval to the year of its ending source date
yearly_data["year"] = yearly_data["source_date"].dt.year

yearly_data["is_candidate"] = (
    yearly_data["calendar_eligibility"].eq("candidate")
)

yearly_data["needs_review"] = (
    yearly_data["calendar_eligibility"].eq("review")
)

yearly_summary = (
    yearly_data.groupby("year")
    .agg(
        first_interval_start=("previous_source_date", "min"),
        last_interval_end=("source_date", "max"),
        total_changes=("change_percent", "count"),
        candidate_intervals=("is_candidate", "sum"),
        review_intervals=("needs_review", "sum"),
        volatility_pct=("change_percent", "std"),
        largest_decline_pct=("change_percent", "min"),
        largest_increase_pct=("change_percent", "max"),
    )
    .reset_index()
)

yearly_summary["coverage_note"] = yearly_summary["year"].map(
    lambda year: (
        "Partial year: through September 30"
        if year == 2026
        else "Within study coverage"
    )
)

display(
    yearly_summary.round({
        "volatility_pct": 4,
        "largest_decline_pct": 4,
        "largest_increase_pct": 4,
    })
)


**Finding:** 2022 was informative but not uniquely extreme. Among complete years, 2023 had slightly higher standard deviation (0.3448% versus 0.3394%); 2025 had the largest observed decline/increase (-1.5036%/+1.5356%). The partial-2026 standard deviation was 0.3623%. These sample differences are not formal significance findings. Cross-year drawdown rankings have not been assessed.

## 10. Check initial training-history availability

**Question:** when is there enough prior candidate history for 250- and 500-change estimates?

Count only changes ending at or before the target interval's starting observation. Full warm-up is required; target changes are excluded. These are retrospective sequence-coverage dates, not verified operational forecast times.

In [ ]:
# Keep original interval changes; do not recompute across exclusions
coverage = (
    interval_calendar_audit
    .sort_values("source_date")
    .reset_index(drop=True)
    .copy()
)

coverage["is_candidate"] = (
    coverage["calendar_eligibility"].eq("candidate")
)

# Number of candidate changes available BEFORE each target interval
coverage["prior_candidate_count"] = (
    coverage["is_candidate"]
    .astype(int)
    .cumsum()
    .shift(1, fill_value=0)
)

coverage_results = []

for window_size in [250, 500]:

    # Only candidate target intervals with enough prior candidate history
    targets = coverage.loc[
        coverage["is_candidate"]
        & (coverage["prior_candidate_count"] >= window_size)
    ]

    if targets.empty:
        coverage_results.append({
            "window_changes": window_size,
            "status": "Insufficient history",
        })
        continue

    first_target = targets.iloc[0]

    # Use changes ending at or before the starting observation; exclude the target.
    training = coverage.loc[
        coverage["is_candidate"]
        & (
            coverage["source_date"]
            <= first_target["previous_source_date"]
        )
    ].tail(window_size)

    training_start = training.iloc[0]["previous_source_date"]
    training_end = training.iloc[-1]["source_date"]

    coverage_results.append({
        "window_changes": window_size,
        "status": "Provisional calendar coverage available",
        "first_training_start": training_start,
        "first_training_end": training_end,
        "training_calendar_days": (
            training_end - training_start
        ).days,
        "first_target_start": first_target["previous_source_date"],
        "first_target_end": first_target["source_date"],
        "candidate_backtest_targets": len(targets),
        "last_target_end": targets.iloc[-1]["source_date"],
    })

window_coverage = pd.DataFrame(coverage_results)

display(window_coverage)


**Finding:** 250 candidates permit a first target of January 19–20, 2021 and 1,390 candidate targets. For 500, the first target is January 20–21, 2022 with 1,140 targets. Compare model variants on common targets; do not compare their different full-history exception rates as if samples were identical. Availability assumptions still apply.

## 11. Closeout diagnostic: zero and repeated changes

**Question:** are unchanged rates or repeated changes a data-quality concern?

Zero means exact equality of recorded numeric endpoints. Nonzero changes are grouped to six decimal places only for this exploratory check; this does not establish exact mathematical equality. Do not round or alter modelling inputs.

In [ ]:
# All original intervals, including those marked for review
repeat_audit = fx_changes.loc[
    fx_changes["change_percent"].notna()
].copy()

repeat_audit["year"] = repeat_audit["source_date"].dt.year

# Exact equality of the two recorded numeric endpoint rates
repeat_audit["is_zero_change"] = (
    repeat_audit["usd_php_rate"]
    == repeat_audit["previous_rate"]
)

# Zero-change frequency per year
zero_summary = (
    repeat_audit.groupby("year")
    .agg(
        total_intervals=("is_zero_change", "size"),
        zero_changes=("is_zero_change", "sum"),
    )
    .reset_index()
)

zero_summary["zero_share_pct"] = (
    zero_summary["zero_changes"]
    / zero_summary["total_intervals"]
    * 100
)

display(zero_summary.round({"zero_share_pct": 4}))

# Group approximately equal nonzero percentage changes for this diagnostic only.
# This is an exploratory grouping, NOT proof of exact equality.
repeat_audit["change_pct_group_6dp"] = (
    repeat_audit["change_percent"].round(6)
)

repeated_changes = (
    repeat_audit.loc[~repeat_audit["is_zero_change"]]
    .groupby("change_pct_group_6dp")
    .agg(
        number_of_intervals=("source_date", "size"),
        first_seen=("source_date", "min"),
        last_seen=("source_date", "max"),
    )
    .reset_index()
)

repeated_changes = (
    repeated_changes.loc[
        repeated_changes["number_of_intervals"] > 1
    ]
    .sort_values(
        ["number_of_intervals", "change_pct_group_6dp"],
        ascending=[False, True],
    )
)

print("Repeated nonzero changes, grouped to 6 decimal places (%):")
display(repeated_changes.head(10))

# Examples of intervals with exactly unchanged recorded rates
print("First 10 zero-change intervals:")
display(
    repeat_audit.loc[
        repeat_audit["is_zero_change"],
        [
            "previous_source_date",
            "source_date",
            "gap_calendar_days",
            "previous_rate",
            "usd_php_rate",
        ],
    ].head(10)
)


**Finding:** ten of 1,647 intervals have unchanged recorded rates (about 0.61%): three in 2020, two in 2022 and five in 2024. Three nonzero six-decimal percentage groups occur twice each; approximate repeats alone are not evidence of duplicated records.

**Bounded source follow-up (separate from the calculation above):** nine observations in the December 16–18, 2020, July 24–26, 2024 and October 23–25, 2024 runs were independently matched through raw, staging and original BSP bulletins. Their USD-row values are respectively 48.0590, 58.3890 and 57.7980 throughout each run. This covers six zero-change intervals; four isolated zero intervals were not included in that bulletin check.

The repetitions were not introduced by raw-to-staging processing. Causes remain unconfirmed. December PDS closing values change despite the repeated USD-row value; July/October PDS closing dates repeat, but PDS is a different field and cannot establish the USD economic fixing date. Preserve these source-confirmed values and do not substitute PDS values. Zero published-series change is not proof of no underlying market movement.

Evidence: [bounded source review](eda_evidence/repeated-runs/REPEATED_RUN_REVIEW.md) and [nine-row reconciliation](eda_evidence/repeated-runs/repeated_run_reconciliation.csv).

## 12. Closeout diagnostic: lag-1 linear dependence

**Question:** are adjacent original changes linearly related?

Use all original intervals, including review intervals, before forming pairs. Filtering first could create false adjacency. The full sample includes cross-year pairs; yearly estimates use within-year pairs only. Correlation is not a direction probability.

In [ ]:
# All original changes, including review intervals
dependence = (
    interval_calendar_audit
    .sort_values("source_date")
    .copy()
)

dependence["year"] = dependence["source_date"].dt.year

# Pair each change with the immediately preceding original change
dependence["previous_change_pct"] = (
    dependence["change_percent"].shift(1)
)

pairs = dependence.dropna(
    subset=["previous_change_pct", "change_percent"]
)

summary_rows = [{
    "period": "Full study sample",
    "number_of_pairs": len(pairs),
    "lag1_autocorrelation": pairs["previous_change_pct"].corr(
        pairs["change_percent"]
    ),
}]

# Within-year pairs only; no pairing across year boundaries here
for year, group in dependence.groupby("year"):
    changes = group["change_percent"].dropna()

    summary_rows.append({
        "period": str(year),
        "number_of_pairs": max(len(changes) - 1, 0),
        "lag1_autocorrelation": changes.autocorr(lag=1),
    })

autocorrelation_summary = pd.DataFrame(summary_rows)

display(
    autocorrelation_summary.round({"lag1_autocorrelation": 4})
)

# Scatter plot of adjacent original changes
fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(
    pairs["previous_change_pct"],
    pairs["change_percent"],
    alpha=0.35,
    s=18,
    color="steelblue",
)

ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)

ax.set_title("Adjacent Original Percentage Changes | Full Study Sample")
ax.set_xlabel("Previous interval change (%)")
ax.set_ylabel("Current interval change (%)")
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()


**Finding:** full-sample lag-1 correlation is 0.1072 across 1,646 adjacent pairs; yearly estimates range from -0.0857 to +0.1935. This indicates weak positive linear dependence overall, not a 10.72% movement probability. No formal significance assessment or cause is established. Weak correlation does not imply independence, and averaging is not established as its cause. Historical simulation pools scenarios rather than modelling their sequence; inspect exception timing and clustering later.

## 13. Closeout diagnostic: all-window calendar composition

**Question:** how much calendar history and what gap mix do fixed-size training windows contain?

Use the previous 250/500 candidate changes for each candidate target. Exclusions make history reach farther back; they never create replacement returns. Calendar spans describe training coverage, not forecast horizons.

In [ ]:
# Original calendar-candidate intervals only
candidates = (
    interval_calendar_audit.loc[
        interval_calendar_audit["calendar_eligibility"].eq("candidate")
    ]
    .sort_values("source_date")
    .reset_index(drop=True)
)

window_rows = []

for window_size in [250, 500]:

    # Each target uses only the candidate changes BEFORE it
    for target_index in range(window_size, len(candidates)):
        training = candidates.iloc[
            target_index - window_size : target_index
        ]

        target = candidates.iloc[target_index]

        training_start = training.iloc[0]["previous_source_date"]
        training_end = training.iloc[-1]["source_date"]

        multi_gap_count = training["gap_calendar_days"].gt(1).sum()

        window_rows.append({
            "window_changes": window_size,
            "target_end": target["source_date"],
            "training_start": training_start,
            "training_end": training_end,
            "training_span_calendar_days": (
                training_end - training_start
            ).days,
            "gap1_count": training["gap_calendar_days"].eq(1).sum(),
            "gap_gt1_count": multi_gap_count,
            "gap_gt1_share_pct": multi_gap_count / window_size * 100,
            "largest_gap_calendar_days": (
                training["gap_calendar_days"].max()
            ),
        })

training_window_audit = pd.DataFrame(window_rows)

window_summary = (
    training_window_audit.groupby("window_changes")
    .agg(
        number_of_windows=("target_end", "size"),
        shortest_span_days=("training_span_calendar_days", "min"),
        median_span_days=("training_span_calendar_days", "median"),
        longest_span_days=("training_span_calendar_days", "max"),
        lowest_gap_gt1_share_pct=("gap_gt1_share_pct", "min"),
        median_gap_gt1_share_pct=("gap_gt1_share_pct", "median"),
        highest_gap_gt1_share_pct=("gap_gt1_share_pct", "max"),
        largest_included_gap_days=("largest_gap_calendar_days", "max"),
    )
    .reset_index()
)

# Each window must contain exactly the required number of intervals
assert (
    training_window_audit["gap1_count"]
    + training_window_audit["gap_gt1_count"]
    == training_window_audit["window_changes"]
).all()

display(window_summary.round({
    "median_span_days": 1,
    "lowest_gap_gt1_share_pct": 2,
    "median_gap_gt1_share_pct": 2,
    "highest_gap_gt1_share_pct": 2,
}))


**Finding:** 250-change windows span 361–388 calendar days (median 375), with gap>1 shares 21.6–25.6% (median 23.2%). For 500, spans are 739–762 days (median 749), with shares 21.8–24.4% (median 23.0%). The longest included endpoint gap is six days. Similar typical gap mixes do not establish equal economic durations or constant risk. The agreed pooled model therefore retains separate gap-group reporting.

## 14. EDA closeout and modelling handoff

**Conclusion:** a positive endpoint result can coexist with large interim declines. P&L, drawdown and volatility complement one another; none alone establishes the position's full risk. 2022 illustrates this clearly but is not uniquely extreme across the sample.

| Evidence | Modelling implication |
|---|---|
| Roughly symmetric 2022 changes with positive excess kurtosis | Examine tail risk without declaring normality or method superiority |
| 1,640 calendar candidates and seven review intervals | Freeze original interval IDs; preserve exclusions and inclusion sensitivity |
| Source-confirmed repeated values in three targeted runs | Retain genuine recorded zeros; disclose unresolved economic freshness |
| Weak positive lag-1 linear dependence | Examine exception timing/clustering; do not assume independence |
| Variable training spans and mixed calendar gaps | Report gap groups and retain the mixed-duration interpretation |

**Agreed exploratory milestone:** coverage, trends, fixed-position P&L, drawdown, illustrative flag, volatility, distribution, yearly comparisons, calendar eligibility, warm-up, repeated values, dependence and all-window calendar composition are documented. The targeted bulletin follow-up is adviser-verified evidence; the EDA calculations were developed and run with guidance. This editorial pass does not establish independent learner mastery.

**Controlling next-phase specification:** [MODELLING_CONTRACT.md v1.0](../MODELLING_CONTRACT.md). Main model: 99% loss quantile using 250 prior eligible changes and `higher`; sensitivities: 500/`higher` and 250/`linear`, evaluated on common targets. Preserve signed VaR, apply strict exceedance, separately audit near-ties, and retain the frozen seven-interval main exclusions and inclusion sensitivity.

**Limits:** underlying market dates, availability timestamps, full-period method continuity, repeated-value causes and historical revisions remain incompletely verified. No confirmed news causation, predictive strategy, tradable P&L, calibrated review threshold, verified market-close one-day VaR or regulatory adequacy is claimed.

**Next:** build the auditable original-interval ledger and eligible rolling-window selection in a separate modelling notebook. VaR implementation, numerical/chronology acceptance checks, stress scenarios and backtesting remain pending. No model result is produced here.
